<a href="https://colab.research.google.com/github/vidhya2026/Duffl_Career_Page/blob/main/Resume_Matcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install libraries


In [2]:
!pip install docx2txt PyPDF2 scikit-learn
!pip install PyMuPDF python-docx scikit-learn pandas openpyxl -q
!pip install PyMuPDF python-docx sentence-transformers scikit-learn pandas openpyxl ipywidgets -q

In [10]:
!pip install PyMuPDF python-docx sentence-transformers scikit-learn pandas openpyxl ipywidgets spacy -q
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 31.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
# ============================================================
# RESUME MATCHER v4 — Sentence Transformers (BERT)
# HR provides JD + threshold + uploads resumes. No hardcoding.
# Uses: all-MiniLM-L6-v2 (BERT) + spaCy en_core_web_sm (NER)
#
# New in v4:
#   - Threshold is set by HR at runtime (not hardcoded)
#   - Candidate Name extracted via spaCy NER + fallback heuristics
#   - Years of Experience extracted from resume text
# ============================================================


# ============================================================
# CELL 1 — Install libraries
# ============================================================

!pip install PyMuPDF python-docx sentence-transformers scikit-learn pandas openpyxl ipywidgets spacy -q
!python -m spacy download en_core_web_sm -q


# ============================================================
# CELL 2 — Imports
# ============================================================

import fitz
import docx
import io, re, datetime
import numpy as np
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output

print("✅ All libraries imported.")


# ============================================================
# CELL 3 — Load models
# ============================================================

print("⏳ Loading BERT sentence-transformer model (first run ~90MB download)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ BERT model loaded (384-dim vectors).")

print("⏳ Loading spaCy NER model...")
nlp = spacy.load("en_core_web_sm")
print("✅ spaCy NER model loaded.")


# ============================================================
# CELL 4 — Text extraction helpers
# ============================================================

def extract_pdf(file_bytes):
    text = ""
    try:
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        for page in doc:
            text += page.get_text()
    except Exception as e:
        text = f"[PDF error: {e}]"
    return text.strip()

def extract_docx(file_bytes):
    text = ""
    try:
        doc = docx.Document(io.BytesIO(file_bytes))
        for para in doc.paragraphs:
            text += para.text + "\n"
    except Exception as e:
        text = f"[DOCX error: {e}]"
    return text.strip()

def extract_text(filename, file_bytes):
    ext = filename.lower().split(".")[-1]
    if ext == "pdf":
        return extract_pdf(file_bytes)
    elif ext in ("docx", "doc"):
        return extract_docx(file_bytes)
    elif ext == "txt":
        return file_bytes.decode("utf-8", errors="ignore")
    return ""

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,\-]', ' ', text)
    return text.strip()


# ============================================================
# CELL 5 — Name extraction
#
# Strategy (in priority order):
#  1. spaCy PERSON entity from first 300 chars
#  2. spaCy PERSON entity from first 2000 chars (full scan)
#  3. Explicit "Name: ..." label pattern
#  4. First line that looks like a human name (heuristic)
#  5. Return "Unknown"
# ============================================================

_RESUME_KEYWORDS = {
    "resume", "curriculum", "vitae", "cv", "summary", "objective",
    "profile", "contact", "email", "phone", "mobile", "address",
    "linkedin", "github", "portfolio", "education", "experience",
    "skills", "projects", "certifications", "languages", "references",
    "declaration", "hobbies", "interests", "achievements", "awards",
    "work", "employment", "career"
}

def _looks_like_name(text: str) -> bool:
    """Returns True if the string looks like a human name."""
    text = text.strip()
    if not text or re.search(r'\d', text):
        return False
    words = text.split()
    if not (2 <= len(words) <= 5):
        return False
    if not all(w[0].isupper() for w in words if len(w) > 0):
        return False
    if any(w.lower() in _RESUME_KEYWORDS for w in words):
        return False
    # Avoid lines that are all caps (section headers like "JOHN DOE SMITH" are fine,
    # but "PROFESSIONAL SUMMARY" should be filtered by keyword check above)
    return True

def extract_name(resume_text: str) -> str:
    """
    Extracts the candidate name from resume text.
    Falls through multiple strategies until a name is found.
    """
    # Strategy 1: spaCy NER on first 300 characters
    first_chunk = resume_text[:300]
    doc = nlp(first_chunk)
    for ent in doc.ents:
        if ent.label_ == "PERSON" and _looks_like_name(ent.text):
            return ent.text.strip()

    # Strategy 2: spaCy NER on first 2000 characters
    full_doc = nlp(resume_text[:2000])
    for ent in full_doc.ents:
        if ent.label_ == "PERSON" and _looks_like_name(ent.text):
            return ent.text.strip()

    # Strategy 3: Explicit "Name:" or "Name -" label
    name_label = re.search(
        r'(?:^|\n)\s*[Nn]ame\s*[:\-]\s*([A-Z][a-zA-Z]+(?:\s[A-Z][a-zA-Z]+){1,3})',
        resume_text[:500]
    )
    if name_label:
        candidate = name_label.group(1).strip()
        if _looks_like_name(candidate):
            return candidate

    # Strategy 4: First line(s) that look like a name
    lines = [ln.strip() for ln in resume_text.splitlines() if ln.strip()]
    for line in lines[:12]:
        # Strip separators and check each segment
        for segment in re.split(r'[\|•,/\\]', line):
            segment = segment.strip()
            if _looks_like_name(segment):
                return segment

    return "Unknown"


# ============================================================
# CELL 6 — Experience extraction
#
# Looks for:
#   "5 years", "3+ years of experience", "2.5 yrs"
#   Date ranges: "Jan 2018 – Present", "2019 - 2022"
# Returns the highest total years found.
# ============================================================

def extract_experience_years(resume_text: str) -> str:
    """
    Extracts years of professional experience from resume text.
    Returns a formatted string like "5 years" or "Not mentioned".
    """
    text = resume_text.lower()
    candidates = []
    current_year = datetime.datetime.now().year

    # Pattern A: explicit "X years [of experience/work/exp]"
    explicit_matches = re.findall(
        r'(\d+\.?\d*)\s*\+?\s*(?:years?|yrs?)(?:\s+of)?(?:\s+(?:experience|exp|work|professional))?',
        text
    )
    for val in explicit_matches:
        try:
            y = float(val)
            if 0 < y <= 50:   # sanity check
                candidates.append(y)
        except:
            pass

    # Pattern B: year ranges like "2018 - 2023" or "2020 – Present"
    date_range_pattern = re.findall(
        r'(?:(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?\s+)?'
        r'(20\d{2}|19\d{2})'
        r'\s*[-–—to]+\s*'
        r'(present|current|now|20\d{2}|19\d{2})',
        text
    )
    total_range_years = 0
    for start_str, end_str in date_range_pattern:
        try:
            start = int(start_str)
            end   = current_year if end_str in ('present', 'current', 'now') else int(end_str)
            diff  = end - start
            if 0 < diff <= 50:
                total_range_years += diff
        except:
            pass

    if total_range_years > 0:
        candidates.append(float(total_range_years))

    if not candidates:
        return "Not mentioned"

    best = max(candidates)
    if best == int(best):
        return f"{int(best)} year{'s' if int(best) != 1 else ''}"
    else:
        return f"{best:.1f} years"


# ============================================================
# CELL 7 — Missing keywords & BERT matching logic
# ============================================================

def get_missing_keywords(jd_text, resume_text, top_n=8):
    jd_clean  = clean_text(jd_text).lower()
    res_clean = clean_text(resume_text).lower()
    try:
        vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=60)
        vec.fit([jd_clean])
        jd_keywords = vec.get_feature_names_out()
    except:
        return []
    resume_words = set(res_clean.split())
    missing = [kw for kw in jd_keywords if not set(kw.split()).intersection(resume_words)]
    return missing[:top_n]


def match_all(jd_text, resumes_dict, threshold):
    """
    Runs BERT semantic matching for all uploaded resumes against the JD.
    Also extracts Name and Experience for each resume.
    Returns a sorted DataFrame.
    """
    jd_clean     = clean_text(jd_text)
    jd_embedding = model.encode([jd_clean], convert_to_numpy=True)

    results = []
    total   = len(resumes_dict)

    for i, (fname, raw_text) in enumerate(resumes_dict.items(), 1):
        print(f"  [{i}/{total}] Processing: {fname}")

        # Extract name and experience
        candidate_name = extract_name(raw_text)
        experience     = extract_experience_years(raw_text)

        resume_clean = clean_text(raw_text)

        if len(resume_clean) < 30:
            results.append({
                "File":           fname,
                "Candidate Name": candidate_name,
                "Experience":     experience,
                "Score (%)":      0.0,
                "Status":         "❌ Not Matched",
                "Issues":         "Could not extract text from file"
            })
            continue

        resume_embedding = model.encode([resume_clean], convert_to_numpy=True)
        score            = cosine_similarity(jd_embedding, resume_embedding)[0][0]
        score_pct        = round(float(score) * 100, 2)

        if score_pct >= threshold:
            status = "✅ Matched"
            issues = "—"
        else:
            status  = "❌ Not Matched"
            missing = get_missing_keywords(jd_text, raw_text)
            issues  = f"Semantic similarity too low ({score_pct}%)"
            if missing:
                issues += f" | Possibly missing: {', '.join(missing)}"

        results.append({
            "File":           fname,
            "Candidate Name": candidate_name,
            "Experience":     experience,
            "Score (%)":      score_pct,
            "Status":         status,
            "Issues":         issues
        })

    df = pd.DataFrame(results)
    df = df.sort_values("Score (%)", ascending=False).reset_index(drop=True)
    return df


# ============================================================
# CELL 8 — STEP 1: HR enters Job Description + Threshold
# ============================================================

print("=" * 65)
print("  RESUME MATCHER v4 — HR INPUT PANEL")
print("=" * 65)
print()
print("STEP 1: Paste the full Job Description and set your match threshold.")
print("        Then click 'Save JD & Continue'.\n")

jd_box = widgets.Textarea(
    placeholder=(
        "Paste the full job description here...\n\n"
        "Example:\nWe are looking for a Python Developer with experience in\n"
        "machine learning, deep learning, NLP, TensorFlow, SQL,\n"
        "REST APIs, AWS, and good communication skills."
    ),
    layout=widgets.Layout(width='100%', height='240px')
)

threshold_slider = widgets.IntSlider(
    value=50,
    min=10, max=90, step=5,
    description='Match threshold %:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='65%')
)

threshold_note = widgets.HTML(
    value=(
        "<div style='font-size:12px; color:#555; margin-top:4px;'>"
        "<b>Tip:</b> BERT semantic scores are generally higher than keyword-based methods.<br>"
        "• <b>50%</b> — Good general starting point (recommended)<br>"
        "• <b>60–70%</b> — Stricter: only close matches pass<br>"
        "• <b>75%+</b> — Very strict: near-identical skill sets required"
        "</div>"
    )
)

save_btn = widgets.Button(
    description='Save JD & Continue ➜',
    button_style='primary',
    layout=widgets.Layout(width='230px')
)

status_out  = widgets.Output()
jd_saved    = {'text': None, 'threshold': 50}

def on_save(b):
    with status_out:
        clear_output()
        txt = jd_box.value.strip()
        if len(txt) < 30:
            print("⚠️  JD is too short. Please paste the full job description.")
            return
        jd_saved['text']      = txt
        jd_saved['threshold'] = threshold_slider.value
        print(f"✅ JD saved — {len(txt)} characters.")
        print(f"✅ Match threshold set to: {threshold_slider.value}%")
        print("➡️  Now run CELL 9 to upload resumes.")

save_btn.on_click(on_save)

display(widgets.VBox([
    widgets.Label("Job Description:"),
    jd_box,
    widgets.HTML("<br>"),
    threshold_slider,
    threshold_note,
    widgets.HTML("<br>"),
    save_btn,
    status_out
]))




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 18.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ All libraries imported.
⏳ Loading BERT sentence-transformer model (first run ~90MB download)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERT model loaded (384-dim vectors).
⏳ Loading spaCy NER model...
✅ spaCy NER model loaded.
  RESUME MATCHER v4 — HR INPUT PANEL

STEP 1: Paste the full Job Description and set your match threshold.
        Then click 'Save JD & Continue'.



In [7]:
# ============================================================
# CELL 9 — STEP 2: HR uploads resumes
# ============================================================

print("STEP 2: Upload candidate resumes.")
print("        Supported formats: PDF, DOCX, TXT")
print("        You can select multiple files at once.\n")

uploaded = files.upload()

resumes_dict = {}
for fname, fbytes in uploaded.items():
    text = extract_text(fname, fbytes)
    resumes_dict[fname] = text
    ok = "✓ text extracted" if len(text) > 30 else "⚠ empty / unreadable"
    print(f"  {fname:50s}  {ok}  ({len(text)} chars)")

print(f"\n Total resumes ready: {len(resumes_dict)}")
print(" Now run CELL 10 to start matching.")


# ============================================================
# CELL 10 — STEP 3: Run BERT matching & show results
# ============================================================

if not jd_saved['text']:
    print("⚠️  No JD found. Run CELL 8 first and click 'Save JD & Continue'.")

elif not resumes_dict:
    print("⚠️  No resumes found. Run CELL 9 first.")

else:
    threshold = jd_saved['threshold']
    print(f"🔄 Running BERT semantic matching (threshold = {threshold}%)...")
    print("   Each resume → 384-dim vector → cosine similarity with JD vector")
    print("   Name extracted via spaCy NER + heuristics")
    print("   Experience extracted via pattern matching\n")

    results_df = match_all(jd_saved['text'], resumes_dict, threshold)

    # ── Results table ─────────────────────────────────────────
    print("\n" + "=" * 70)
    print("  RESULTS")
    print("=" * 70)
    pd.set_option('display.max_colwidth', 80)
    pd.set_option('display.width', 180)
    display(results_df)

    matched     = (results_df["Status"] == "✅ Matched").sum()
    not_matched = len(results_df) - matched
    print(f"\n📊 Summary: {matched} matched,  {not_matched} not matched  "
          f"(threshold = {threshold}%,  total = {len(results_df)})")

    # ── Detailed breakdown ────────────────────────────────────
    print("\n" + "─" * 70)
    print("DETAILED BREAKDOWN")
    print("─" * 70)
    for _, row in results_df.iterrows():
        print(f"\n📄 File           : {row['File']}")
        print(f"   Candidate Name : {row['Candidate Name']}")
        print(f"   Experience     : {row['Experience']}")
        print(f"   BERT Score     : {row['Score (%)']}%")
        print(f"   Status         : {row['Status']}")
        if row['Status'] == "❌ Not Matched":
            print(f"   Issues         : {row['Issues']}")

    # ── Download results ──────────────────────────────────────
    csv_path  = "resume_results_bert_v4.csv"
    xlsx_path = "resume_results_bert_v4.xlsx"
    results_df.to_csv(csv_path, index=False)
    results_df.to_excel(xlsx_path, index=False)

    print("\n📥 Downloading results...")
    files.download(csv_path)
    files.download(xlsx_path)
    print("✅ All done!")

STEP 2: Upload candidate resumes.
        Supported formats: PDF, DOCX, TXT
        You can select multiple files at once.



Saving Naukri_R.SURYAPRAKASH[3y_5m].pdf to Naukri_R.SURYAPRAKASH[3y_5m].pdf
  Naukri_R.SURYAPRAKASH[3y_5m].pdf                    ✓ text extracted  (5296 chars)

 Total resumes ready: 1
 Now run CELL 10 to start matching.
🔄 Running BERT semantic matching (threshold = 50%)...
   Each resume → 384-dim vector → cosine similarity with JD vector
   Name extracted via spaCy NER + heuristics
   Experience extracted via pattern matching

  [1/1] Processing: Naukri_R.SURYAPRAKASH[3y_5m].pdf

  RESULTS


,File,Candidate Name,Experience,Score (%),Status,Issues
0,Naukri_R.SURYAPRAKASH[3y_5m].pdf,SURYA PRAKASH R,4 years,55.34,✅ Matched,—



📊 Summary: 1 matched,  0 not matched  (threshold = 50%,  total = 1)

──────────────────────────────────────────────────────────────────────
DETAILED BREAKDOWN
──────────────────────────────────────────────────────────────────────

📄 File           : Naukri_R.SURYAPRAKASH[3y_5m].pdf
   Candidate Name : SURYA PRAKASH R
   Experience     : 4 years
   BERT Score     : 55.34%
   Status         : ✅ Matched

📥 Downloading results...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All done!


In [11]:
!pkill ngrok

In [13]:
# ============================================================
# RESUME MATCHER — Flask API for Google Colab
# Run this in Colab. It exposes a public URL via ngrok.
# Your C# ASP.NET MVC app calls this URL.
#
# Changes:
#   - Threshold is now sent by HR as a form field (not hardcoded)
#   - Extracts candidate Name using spaCy NER + fallback heuristics
#   - Extracts Years of Experience from resume text
# ============================================================


# ============================================================
# CELL 1 — Install everything
# ============================================================

!pip install PyMuPDF python-docx sentence-transformers scikit-learn flask flask-cors pyngrok spacy -q
!python -m spacy download en_core_web_sm -q


# ============================================================
# CELL 2 — Imports
# ============================================================

import fitz
import docx
import io, re
import numpy as np
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading

print("✅ All imports done.")


# ============================================================
# CELL 3 — Load models (BERT + spaCy)
# ============================================================

print("⏳ Loading BERT model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ BERT model ready.")

print("⏳ Loading spaCy NER model...")
nlp = spacy.load("en_core_web_sm")
print("✅ spaCy NER model ready.")


# ============================================================
# CELL 4 — Text extraction helpers
# ============================================================

def extract_pdf(file_bytes):
    text = ""
    try:
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        for page in doc:
            text += page.get_text()
    except Exception as e:
        text = ""
    return text.strip()

def extract_docx(file_bytes):
    text = ""
    try:
        doc = docx.Document(io.BytesIO(file_bytes))
        for para in doc.paragraphs:
            text += para.text + "\n"
    except:
        text = ""
    return text.strip()

def extract_text(filename, file_bytes):
    ext = filename.lower().split(".")[-1]
    if ext == "pdf":
        return extract_pdf(file_bytes)
    elif ext in ("docx", "doc"):
        return extract_docx(file_bytes)
    elif ext == "txt":
        return file_bytes.decode("utf-8", errors="ignore")
    return ""

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,\-]', ' ', text)
    return text.strip()


# ============================================================
# CELL 5 — Name extraction logic
#
#  Strategy (in order):
#  1. spaCy PERSON entity from the first 300 characters
#  2. If not found in first 300 chars, try full text spaCy pass
#  3. Fallback: first non-empty line that looks like a name
#     (2-4 words, all capitalized, no digits, no keywords)
#  4. Fallback: regex for "Name: John Doe" pattern
#  5. If still not found, return "Unknown"
# ============================================================

# Common resume section keywords to ignore during name detection
_RESUME_KEYWORDS = {
    "resume", "curriculum", "vitae", "cv", "summary", "objective",
    "profile", "contact", "email", "phone", "mobile", "address",
    "linkedin", "github", "portfolio", "education", "experience",
    "skills", "projects", "certifications", "languages", "references",
    "declaration", "hobbies", "interests", "achievements", "awards"
}

def _looks_like_name(text: str) -> bool:
    """
    Returns True if the string looks like a person's name:
    - 2 to 5 words
    - Each word starts with a capital letter
    - No digits
    - Not a known resume section keyword
    """
    text = text.strip()
    if not text or re.search(r'\d', text):
        return False
    words = text.split()
    if not (2 <= len(words) <= 5):
        return False
    if not all(w[0].isupper() for w in words if w):
        return False
    if any(w.lower() in _RESUME_KEYWORDS for w in words):
        return False
    return True

def extract_name(resume_text: str) -> str:
    """
    Extracts the candidate's name from resume text.
    Uses spaCy NER first, then falls back to heuristic rules.
    """
    # ── Strategy 1: spaCy on first 300 chars (name is usually at top) ──
    first_chunk = resume_text[:300]
    doc = nlp(first_chunk)
    for ent in doc.ents:
        if ent.label_ == "PERSON" and _looks_like_name(ent.text):
            return ent.text.strip()

    # ── Strategy 2: spaCy on full text ──
    full_doc = nlp(resume_text[:2000])   # limit for speed
    for ent in full_doc.ents:
        if ent.label_ == "PERSON" and _looks_like_name(ent.text):
            return ent.text.strip()

    # ── Strategy 3: Explicit "Name:" label in resume ──
    name_label_match = re.search(
        r'(?:Name\s*[:\-]\s*)([A-Z][a-zA-Z]+(?:\s[A-Z][a-zA-Z]+){1,3})',
        resume_text[:500]
    )
    if name_label_match:
        candidate = name_label_match.group(1).strip()
        if _looks_like_name(candidate):
            return candidate

    # ── Strategy 4: First non-empty line that looks like a name ──
    lines = [ln.strip() for ln in resume_text.splitlines() if ln.strip()]
    for line in lines[:10]:   # check first 10 lines only
        # Strip common noise like phone numbers, emails
        clean_line = re.sub(r'[\|•|]', ' ', line).strip()
        # Try each segment split by common separators
        for segment in re.split(r'[\|•,/]', clean_line):
            segment = segment.strip()
            if _looks_like_name(segment):
                return segment

    # ── Strategy 5: Give up ──
    return "Unknown"


# ============================================================
# CELL 6 — Experience extraction logic
#
#  Looks for patterns like:
#    "5 years of experience", "3+ years", "2.5 years"
#    Date ranges: "Jan 2020 – Present", "2018 - 2023"
#  Returns the highest number found (most representative).
# ============================================================

def extract_experience_years(resume_text: str) -> str:
    """
    Extracts years of experience from resume text.
    Returns a string like "5 years" or "Not mentioned".
    """
    text = resume_text.lower()
    candidates = []

    # ── Pattern A: explicit "X years [of experience]" ──
    explicit = re.findall(
        r'(\d+\.?\d*)\s*\+?\s*years?(?:\s+of)?(?:\s+(?:experience|exp|work))?',
        text
    )
    for val in explicit:
        try:
            candidates.append(float(val))
        except:
            pass

    # ── Pattern B: date ranges (e.g. "2018 – 2023" or "2020 - present") ──
    import datetime
    current_year = datetime.datetime.now().year

    # "YYYY - YYYY" or "YYYY – Present"
    date_ranges = re.findall(
        r'((?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)?\.?\s*(\d{4})'
        r'\s*[-–—to]+\s*'
        r'(present|current|now|(\d{4})))',
        text
    )
    total_from_ranges = 0
    for match in date_ranges:
        start_year = int(match[1])
        end_str    = match[2].strip()
        if end_str in ('present', 'current', 'now'):
            end_year = current_year
        else:
            try:
                end_year = int(match[3]) if match[3] else current_year
            except:
                end_year = current_year
        diff = end_year - start_year
        if 0 < diff <= 50:   # sanity check
            total_from_ranges += diff

    if total_from_ranges > 0:
        candidates.append(float(total_from_ranges))

    if not candidates:
        return "Not mentioned"

    # Return the maximum found value (most likely total experience)
    best = max(candidates)
    # Format nicely
    if best == int(best):
        return f"{int(best)} year{'s' if best != 1 else ''}"
    else:
        return f"{best} years"


# ============================================================
# CELL 7 — Missing keywords & core matching logic
# ============================================================

def get_missing_keywords(jd_text, resume_text, top_n=8):
    jd_clean  = clean_text(jd_text).lower()
    res_clean = clean_text(resume_text).lower()
    try:
        vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=60)
        vec.fit([jd_clean])
        jd_keywords = vec.get_feature_names_out()
    except:
        return []
    resume_words = set(res_clean.split())
    missing = [kw for kw in jd_keywords if not set(kw.split()).intersection(resume_words)]
    return missing[:top_n]


def match_resume(jd_text, resume_text, filename, threshold):
    """
    Matches one resume against the JD.
    Also extracts candidate name and experience years.
    threshold is passed per-request from HR.
    """
    jd_clean     = clean_text(jd_text)
    resume_clean = clean_text(resume_text)

    # Extract name and experience from raw (un-cleaned) text
    candidate_name = extract_name(resume_text)
    experience     = extract_experience_years(resume_text)

    if len(resume_clean) < 30:
        return {
            "file":            filename,
            "candidate_name":  candidate_name,
            "experience":      experience,
            "score":           0.0,
            "status":          "Not Matched",
            "issues":          "Could not extract text from file"
        }

    jd_emb    = model.encode([jd_clean],     convert_to_numpy=True)
    res_emb   = model.encode([resume_clean], convert_to_numpy=True)
    score     = cosine_similarity(jd_emb, res_emb)[0][0]
    score_pct = round(float(score) * 100, 2)

    if score_pct >= threshold:
        status = "Matched"
        issues = ""
    else:
        status  = "Not Matched"
        missing = get_missing_keywords(jd_text, resume_text)
        issues  = f"Low similarity score ({score_pct}%)"
        if missing:
            issues += f". Possibly missing: {', '.join(missing)}"

    return {
        "file":           filename,
        "candidate_name": candidate_name,
        "experience":     experience,
        "score":          score_pct,
        "status":         status,
        "issues":         issues
    }




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ All imports done.
⏳ Loading BERT model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERT model ready.
⏳ Loading spaCy NER model...
✅ spaCy NER model ready.


In [1]:
import os
os.system("fuser -k 5000/tcp")

256

In [3]:
!pip install flask flask-cors pyngrok -q

In [10]:
!pip install pyngrok -q

In [9]:
from flask import Flask, request, jsonify
from flask_cors import CORS


In [12]:
!pip install pyngrok -q

from pyngrok import ngrok

In [16]:
# ============================================================
# CELL 8 — Flask API

import threading
from flask import Flask
app = Flask(__name__)
CORS(app)

@app.route('/match', methods=['POST'])
def match():
    """
    Expects (multipart/form-data):
      - Form field  : 'jd'         (job description text)
      - Form field  : 'threshold'  (integer 10–90, e.g. "50")
      - Form files  : 'resumes'    (one or more resume files: PDF / DOCX / TXT)

    Returns JSON:
      {
        "threshold_used": 50,
        "results": [
          {
            "file":           "john.pdf",
            "candidate_name": "John Smith",
            "experience":     "5 years",
            "score":          72.4,
            "status":         "Matched",
            "issues":         ""
          },
          ...
        ],
        "summary": {
          "total": 2,
          "matched": 1,
          "not_matched": 1
        }
      }
    """
    try:
        # ── Job description ──
        jd_text = request.form.get('jd', '').strip()
        if len(jd_text) < 10:
            return jsonify({"error": "Job description is too short or missing."}), 400

        # ── Threshold (sent by HR, defaults to 50 if missing) ──
        try:
            threshold = int(request.form.get('threshold', 50))
            threshold = max(10, min(90, threshold))   # clamp 10–90
        except (ValueError, TypeError):
            threshold = 50

        # ── Resumes ──
        resume_files = request.files.getlist('resumes')
        if not resume_files:
            return jsonify({"error": "No resume files uploaded."}), 400

        results = []
        for f in resume_files:
            file_bytes = f.read()
            raw_text   = extract_text(f.filename, file_bytes)
            result     = match_resume(jd_text, raw_text, f.filename, threshold)
            results.append(result)

        # Sort by score descending
        results.sort(key=lambda x: x['score'], reverse=True)

        matched     = sum(1 for r in results if r['status'] == 'Matched')
        not_matched = len(results) - matched

        return jsonify({
            "threshold_used": threshold,
            "results": results,
            "summary": {
                "total":       len(results),
                "matched":     matched,
                "not_matched": not_matched
            }
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route('/health', methods=['GET'])
def health():
    return jsonify({
        "status": "running",
        "model":  "all-MiniLM-L6-v2",
        "note":   "Pass 'threshold' as a form field (10–90). Defaults to 50."
    })


# ============================================================
# CELL 9 — Start Flask + expose via ngrok
# ============================================================

# NOTE: Sign up free at https://ngrok.com and paste your token below
NGROK_AUTH_TOKEN = "3BU7PRMRwFwF4CafRtcEecHDfTg_2r7LLmb2FueGYjGonspVd"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

def run_flask():
    app.run(port=5000)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()

public_url = ngrok.connect(5000)
print("=" * 60)
print("  ✅  Flask API is LIVE!")
print(f"  Public URL  : {public_url}")
print()
print("  POST to  :  {public_url}/match")
print()
print("  Form fields required:")
print("    jd         — job description text")
print("    threshold  — match % (10–90), e.g. 50")
print("    resumes    — one or more PDF / DOCX / TXT files")
print("=" * 60)

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


  ✅  Flask API is LIVE!
  Public URL  : NgrokTunnel: "https://nonappreciative-candi-unlitigated.ngrok-free.dev" -> "http://localhost:5000"

  POST to  :  {public_url}/match

  Form fields required:
    jd         — job description text
    threshold  — match % (10–90), e.g. 50
    resumes    — one or more PDF / DOCX / TXT files
